# 05 · KV cache memory calculator

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path step 4. Sources: model config.json files; NVIDIA datasheets.

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Compute KV cache size for real models, find how many sequences fit on a GPU, and see how the
attention variant and the KV dtype change the answer.

## Theory and equations
Per token, a standard attention layer caches one key and one value per KV head:

$$m_{\text{tok}} = L \cdot 2 \cdot H_{kv} \cdot d_h \cdot s, \qquad M_{\text{KV}} = m_{\text{tok}}\sum_i T_i.$$

MLA caches a latent plus a shared RoPE key, with no separate V: $m_{\text{tok}} = L (d_c + d_R) s$.

## Diagram
```
HBM (80 GB) = [ weights 16 GB ][ reserve 8 GB ][ KV cache: 56 GB = 427,246 tokens of Llama-3-8B BF16 ]
```

## Implementation

In [ ]:
from inference_lab import kv_math as K
for m in K.MODELS.values():
    b = K.kv_bytes_per_token(m)
    print(f"{m.name:12s} {m.attention:4s}  {b/1024:7.1f} KiB/token   {b*32768/2**30:6.2f} GiB per 32K sequence")

## Experiment: capacity on an H100 vs context length and KV dtype (*theoretical*)

In [ ]:
ctx = [2048, 4096, 8192, 16384, 32768, 65536, 131072]
fig, ax = plt.subplots()
for dtype, col in [("bf16", C["memory"]), ("fp8", C["compute"])]:
    n = K.max_concurrent_tokens(K.LLAMA3_8B, K.H100_SXM, kv_dtype=dtype)
    seqs = [n // c for c in ctx]
    ax.plot(ctx, seqs, "o-", color=col, label=f"Llama-3-8B, KV {dtype}")
    print(dtype, dict(zip(ctx, seqs)))
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("context per sequence (tokens)"); ax.set_ylabel("concurrent sequences on 1 x H100")
ax.legend(); plt.show()

## Results: the same model under four attention variants (*theoretical*)

In [ ]:
for v in ("mha", "gqa", "mqa", "mla"):
    b = K.kv_bytes_per_token_variant(K.LLAMA3_8B, v)
    print(f"{v:4s} {b/1024:6.1f} KiB/token  ->  {int(56e9 // (b*32768)):4d} x 32K sequences in 56 GB")

## Interpretation
Capacity falls as $1/T$: every doubling of context halves concurrency. FP8 KV doubles it back.
The variant matters more than the dtype: MHA → GQA-4 is 4×, GQA → MQA another 8×.

## Limitations
Ignores fragmentation (step 6), activation peaks beyond the flat 10% reserve, and per-GPU
replication under tensor parallelism.

## Conclusion
At long context, serving is capacity-bound before it is compute-bound. The cheapest levers are
the KV dtype and choosing a model with an efficient attention variant.